# 03 - Modelo de propensión de adopción

**Qué calcula este notebook:** para cada cliente, un puntaje de probabilidad (entre 0 y 1)
de que le interese adoptar la nueva App de inversiones.

**Cómo lo hace:** como la App todavía no existe, no hay ningún cliente que ya la haya
"adoptado" para enseñarle al modelo cómo se ve eso. Se usa como aproximación la tenencia
de **Invesbot** (el producto de inversión digital más parecido a la nueva App) — se
entrena un modelo que aprende "qué perfil de cliente se parece a quien ya tiene Invesbot",
usando el resto de información del cliente (demografía, otros productos, ingreso
estimado), **sin usar nada de Invesbot como insumo** — eso sería hacer trampa (se llama
*data leakage*: usar la misma respuesta que se quiere predecir como uno de los datos de
entrada).

Se usa **regresión logística**: un modelo simple donde cada característica del cliente
suma o resta puntos a la probabilidad final — fácil de explicar e interpretar tanto para
audiencia técnica como de negocio.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

pd.set_option('display.width', 120)
pd.options.display.float_format = '{:,.2f}'.format

cliente_360 = pd.read_csv('../04_Resultados/cliente_360.csv')
print('Clientes:', len(cliente_360))

Clientes: 859796


## 1. Definir el target (lo que el modelo va a aprender a predecir)

In [2]:
target = cliente_360['tiene_invesbot'].astype(int)
print('Clientes con Invesbot (target=1):', target.sum())
print('Clientes sin Invesbot (target=0):', (target == 0).sum())
print(f'Porcentaje de casos positivos: {target.mean()*100:.2f}%')

Clientes con Invesbot (target=1): 5213
Clientes sin Invesbot (target=0): 854583
Porcentaje de casos positivos: 0.61%


**Ojo con este número:** solo el ~0.6% de los clientes tiene Invesbot. Esto se llama
*desbalance de clases* — hay muchísimos más "no" que "sí". Más adelante esto tiene una
consecuencia importante en cómo entrenamos y cómo evaluamos el modelo (secciones 5 y 6).

## 2. Elegir las variables (features) que va a usar el modelo

Usamos toda la información del cliente **excepto** cualquier columna relacionada con
Invesbot (`saldo_invesbot`, `tiene_invesbot`) — esas no pueden entrar como insumo, porque
son literalmente la respuesta que estamos tratando de predecir.

In [3]:
columnas_categoricas = ['grupo_edad', 'desc_genero', 'desc_segmento', 'desc_tipo_de_vivienda']

columnas_numericas = [
    'ingresos_mensuales', 'total_egresos_mensuales', 'total_activos',
    'total_pasivos', 'total_patrimonio',
    'saldo_aho_cte', 'saldo_bolsillos', 'saldo_fiducuenta', 'saldo_cdt_inv_virtual',
    'estimador_ingreso',
]

columnas_si_no = ['tiene_aho_cte', 'tiene_bolsillos', 'tiene_fiducuenta', 'tiene_cdt_inv_virtual']

datos_modelo = cliente_360[columnas_categoricas + columnas_numericas + columnas_si_no].copy()
datos_modelo.shape

(859796, 18)

## 3. Rellenar el estimador de ingreso faltante

En el notebook 01 dejamos `estimador_ingreso` con vacíos a propósito (rellenarlo con 0
hubiera sido decir "esta persona no gana nada", que es falso). Pero un modelo de regresión
logística sí necesita un número en cada celda, no puede trabajar con vacíos. Aquí es el
momento correcto para decidir qué poner: usamos la mediana (el valor típico) de los
clientes que sí tienen el dato, y guardamos aparte si el dato faltaba o no — por si el
hecho de no tenerlo también dice algo del cliente (por ejemplo, ser un cliente más nuevo).

In [4]:
datos_modelo['tiene_estimador_ingreso'] = datos_modelo['estimador_ingreso'].notna().astype(int)

mediana_estimador = datos_modelo['estimador_ingreso'].median()
datos_modelo['estimador_ingreso'] = datos_modelo['estimador_ingreso'].fillna(mediana_estimador)

print('Vacios restantes en estimador_ingreso:', datos_modelo['estimador_ingreso'].isnull().sum())

Vacios restantes en estimador_ingreso: 0


## 4. Convertir categorías a números (0/1)

Un modelo solo entiende números, no texto como `"personal"` o `"36-49"`. Convertimos cada
categoría en columnas de sí/no (1/0) — a esto se le llama *codificación dummy*. Por cada
variable categórica dejamos una categoría de referencia por fuera (por ejemplo, si
`desc_segmento` tiene "personal", "plus" y "preferencial", dejamos "personal" como base y
el modelo interpreta "plus" y "preferencial" como diferencias respecto a esa base).

In [5]:
X = pd.get_dummies(datos_modelo, columns=columnas_categoricas, drop_first=True)
print('Numero de columnas despues de la codificacion:', X.shape[1])
X.head()

Numero de columnas despues de la codificacion: 28


,ingresos_mensuales,total_egresos_mensuales,total_activos,total_pasivos,total_patrimonio,saldo_aho_cte,saldo_bolsillos,saldo_fiducuenta,saldo_cdt_inv_virtual,estimador_ingreso,...,grupo_edad_65+,desc_genero_masculino,desc_genero_no binario,desc_genero_no informa,desc_genero_trans,desc_segmento_plus,desc_segmento_preferencial,desc_tipo_de_vivienda_FAMILIAR,desc_tipo_de_vivienda_NO INFORMA,desc_tipo_de_vivienda_PROPIA
0,"29,239,444.00","30,000,000.00","145,047,043,000.00","6,356,702,000.00","105,422,323.00",0.00,0.00,0.00,0.00,"2,322,454.50",...,True,True,False,False,False,False,True,False,True,False
1,"31,024,544.00","10,000,000.00","1,133,345,000.00","83,072,000.00","1,050,273,000.00",0.00,0.00,0.00,0.00,"2,322,454.50",...,False,True,False,False,False,False,True,False,False,True
2,"2,834,000.00","500,000.00","1,073,787,017.00",0.00,"343,000,000.00",0.00,0.00,0.00,"450,267,852.65","2,322,454.50",...,True,True,False,False,False,False,True,False,False,True
3,"28,035,850.00",0.00,"175,000,000.00",0.00,"55,000,000.00",0.00,0.00,0.00,0.00,"2,322,454.50",...,True,False,False,False,False,False,True,False,False,True
4,"3,846,205.00","2,500,000.00","758,000,000.00",0.00,"758,000,000.00",0.00,0.00,0.00,0.00,"2,322,454.50",...,False,False,False,False,False,False,True,True,False,False


## 5. Separar en entrenamiento y prueba

Dividimos los clientes en dos grupos: uno para que el modelo **aprenda** (entrenamiento) y
otro que el modelo **nunca ve durante el aprendizaje**, para comprobar honestamente qué
tan bien le va con datos nuevos (prueba). Como el grupo que tiene Invesbot es muy pequeño
(~0.6%), pedimos que esa proporción se mantenga igual en ambos grupos (`stratify`) — si no,
podríamos terminar con muy pocos o ningún caso positivo en alguno de los dos.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, target, test_size=0.25, random_state=42, stratify=target
)

print('Clientes en entrenamiento:', len(X_train))
print('Clientes en prueba:', len(X_test))
print('Porcentaje con Invesbot en entrenamiento:', f'{y_train.mean()*100:.2f}%')
print('Porcentaje con Invesbot en prueba:', f'{y_test.mean()*100:.2f}%')

Clientes en entrenamiento: 644847
Clientes en prueba: 214949
Porcentaje con Invesbot en entrenamiento: 0.61%
Porcentaje con Invesbot en prueba: 0.61%


## 6. Poner las variables numéricas en la misma escala

Variables como `total_patrimonio` (en cientos de millones) y `tiene_aho_cte` (0 o 1) están
en escalas muy distintas. Sin ajustar esto, la regresión logística le daría más peso a las
variables con números más grandes, no porque importen más, sino solo por su tamaño.
Estandarizamos las columnas numéricas para que todas queden en una escala comparable.

**Importante:** calculamos el ajuste de escala (media y desviación) solo con los datos de
entrenamiento, y lo aplicamos igual a los de prueba — así no le "copiamos" al modelo
ninguna información de los datos que se supone no ha visto todavía.

In [7]:
escalador = StandardScaler()

X_train_escalado = X_train.copy()
X_test_escalado = X_test.copy()

X_train_escalado[columnas_numericas] = escalador.fit_transform(X_train[columnas_numericas])
X_test_escalado[columnas_numericas] = escalador.transform(X_test[columnas_numericas])

## 7. Entrenar el modelo

Usamos `class_weight='balanced'`: le decimos al modelo que le dé más importancia a los
pocos clientes que sí tienen Invesbot. Sin esto, como son tan pocos (~0.6%), el modelo
podría "hacer trampa" y simplemente predecir "nadie adopta" — y aun así acertaría casi
siempre, sin servir para nada.

In [8]:
modelo = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
modelo.fit(X_train_escalado, y_train)
print('Modelo entrenado.')

Modelo entrenado.


## 8. ¿Qué tan bien funciona el modelo?

**No basta con mirar el "accuracy" (% de aciertos totales).** Como solo el 0.6% de los
clientes tiene Invesbot, un modelo tonto que dijera "nadie va a adoptar" acertaría el
99.4% de las veces — y sería completamente inútil para el negocio. Por eso miramos otras
métricas, enfocadas en qué tan bien identifica el modelo a los verdaderos candidatos
(clase 1):

- **Recall (de la clase 1):** de todos los que sí tienen Invesbot, ¿a cuántos los detectó
  el modelo? (si es alto, el modelo no se le escapan los candidatos reales)
- **Precision (de la clase 1):** de todos los que el modelo marcó como candidatos, ¿cuántos
  realmente tenían Invesbot? (con clases tan desbalanceadas, es normal que este número sea
  más bajo — el modelo prefiere "avisar de más" a "dejar pasar" candidatos)
- **ROC-AUC:** un número entre 0.5 (el modelo no distingue mejor que tirar una moneda) y
  1.0 (distingue perfecto) que resume qué tan bien separa a quienes probablemente
  adoptarían de quienes no.

In [9]:
predicciones = modelo.predict(X_test_escalado)
probabilidades = modelo.predict_proba(X_test_escalado)[:, 1]

print('--- Matriz de confusion ---')
print(confusion_matrix(y_test, predicciones))
print()
print('--- Reporte de clasificacion ---')
print(classification_report(y_test, predicciones, target_names=['No tiene Invesbot', 'Tiene Invesbot']))
print()
print('ROC-AUC:', roc_auc_score(y_test, probabilidades))

--- Matriz de confusion ---
[[172877  40769]
 [   213   1090]]

--- Reporte de clasificacion ---


                   precision    recall  f1-score   support

No tiene Invesbot       1.00      0.81      0.89    213646
   Tiene Invesbot       0.03      0.84      0.05      1303

         accuracy                           0.81    214949
        macro avg       0.51      0.82      0.47    214949
     weighted avg       0.99      0.81      0.89    214949




ROC-AUC: 0.8977755458066211


**Interpretación de los resultados:**

El **recall** de "Tiene Invesbot" sale alto (el modelo detecta a la mayoría de los
verdaderos casos) pero la **precisión** sale muy baja. Esto no significa que el modelo
esté mal — es una consecuencia directa de dos decisiones tomadas a propósito:

1. Se usó `class_weight='balanced'` (sección 7) para que el modelo no ignore a los pocos
   clientes que sí tienen Invesbot.
2. Se está usando el punto de corte por defecto (0.5) para decidir "sí candidato / no
   candidato" en esta tabla de métricas.

Como los casos reales son solo el 0.6% de la base, con ese punto de corte el modelo
termina marcando a muchas más personas como "candidatas" de las que realmente tienen
Invesbot — de ahí la precisión baja. **Esto no es un problema para el uso que se le da al
modelo:** no se usa un corte fijo de sí/no para decidir a quién contactar. Se usa el
**puntaje de probabilidad tal cual** (columna `probabilidad_adopcion`, sección 10) para
**ordenar** a todos los clientes de mayor a menor probabilidad y priorizar desde arriba —
y para eso, lo que importa es el **ROC-AUC** (qué tan bien ordena el modelo a los
candidatos reales por encima de los que no lo son), que sí sale en un rango bueno
(bastante por encima de 0.5).

## 9. ¿Qué variables pesan más en la decisión del modelo?

Esta es la gran ventaja de la regresión logística: cada variable tiene un coeficiente que
se puede leer directamente. Coeficiente positivo = esa característica **sube** la
probabilidad de tener Invesbot; negativo = la **baja**. Miramos las 10 variables con más
peso en cada sentido.

In [10]:
coeficientes = pd.DataFrame({
    'variable': X_train_escalado.columns,
    'coeficiente': modelo.coef_[0]
}).sort_values('coeficiente', ascending=False)

print('--- Variables que MAS suben la probabilidad ---')
print(coeficientes.head(10).to_string(index=False))
print()
print('--- Variables que MAS bajan la probabilidad ---')
print(coeficientes.tail(10).to_string(index=False))

--- Variables que MAS suben la probabilidad ---
                      variable  coeficiente
       tiene_estimador_ingreso         3.03
         tiene_cdt_inv_virtual         1.37
                 tiene_aho_cte         1.17
              tiene_fiducuenta         1.04
               tiene_bolsillos         0.99
            desc_segmento_plus         0.95
    desc_segmento_preferencial         0.88
         desc_genero_masculino         0.81
        desc_genero_no binario         0.62
desc_tipo_de_vivienda_FAMILIAR         0.35

--- Variables que MAS bajan la probabilidad ---
              variable  coeficiente
 saldo_cdt_inv_virtual        -0.01
desc_genero_no informa        -0.02
      total_patrimonio        -0.03
         total_pasivos        -0.03
         saldo_aho_cte        -0.04
       saldo_bolsillos        -0.10
      grupo_edad_26-35        -0.28
      grupo_edad_36-49        -0.61
      grupo_edad_50-65        -1.07
        grupo_edad_65+        -1.65


## 10. Calcular la probabilidad de adopción para TODOS los clientes

Hasta ahora solo evaluamos el modelo en el grupo de prueba (para saber si funciona bien).
Ahora lo aplicamos a **toda la base** — incluyendo a los clientes que nunca tuvieron
Invesbot — para obtener el puntaje de propensión que se va a usar en el dimensionamiento
del negocio (notebook 05) y en el tablero final.

In [11]:
X_completo_escalado = X.copy()
X_completo_escalado[columnas_numericas] = escalador.transform(X[columnas_numericas])

cliente_360['probabilidad_adopcion'] = modelo.predict_proba(X_completo_escalado)[:, 1]

cliente_360[['numero_id', 'tiene_invesbot', 'probabilidad_adopcion']].sort_values(
    'probabilidad_adopcion', ascending=False
).head(10)

,numero_id,tiene_invesbot,probabilidad_adopcion
828317,-6311229477587712696,False,1.00
828817,636296413544419446,True,1.00
101832,4606201845132161518,False,1.00
822266,-1877552278594685189,True,1.00
4896,-4232095895755647863,False,1.00
823820,-2913746736303330051,True,1.00
824841,-2794453712271137334,False,1.00
832540,212856704848677914,False,1.00
1977,1455836469812299189,False,1.00
829803,-5460472101654469745,False,1.00


## 11. Guardar el resultado

Guardamos solo lo nuevo que aporta este notebook (el identificador del cliente y su
probabilidad calculada) — el notebook 05 lo va a juntar con el resultado del modelo de
monto potencial y el de segmentación para armar la tabla final.

In [12]:
resultado = cliente_360[['numero_id', 'probabilidad_adopcion']]
resultado.to_csv('../04_Resultados/propension.csv', index=False)
print('Guardado:', len(resultado), 'clientes con su probabilidad de adopcion')

Guardado: 859796 clientes con su probabilidad de adopcion
